# weight-decay-decoupled — worked example 3: Show cumulative weight shrinkage over 10 AdamW steps

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-decoupled`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The decoupled weight decay term `p *= (1 - lr * wd)` applies a multiplicative shrinkage to the parameter at every step, independent of whether the gradient is large or small. After T steps, even with zero gradient, the parameter decays toward zero geometrically. This is distinct from the gradient-driven Adam update and is the primary regularization mechanism in AdamW.

## Worked solution

**Step 1 — simulate a regime with near-zero gradient.**
We set the gradient to nearly zero so the Adam update contributes minimally. The dominant effect is then the decoupled weight decay term, making it easy to observe.

**Step 2 — run the loop.**
For each step from 1 to T, we apply the full AdamW update in order: decay, moment update, bias correction, Adam step. We record `p` after each step.

**Step 3 — compare against expected geometric decay.**
With a very small gradient, the parameter should decrease approximately as `p0 * (1 - lr*wd)^T` after T steps. We compare the actual trajectory against this expected geometric baseline to confirm the decay mechanism is working correctly.

In [ ]:
import torch as t

t.manual_seed(77)
T = 10
lr, beta1, beta2, eps, wd = 1e-2, 0.9, 0.999, 1e-8, 0.05

p = t.tensor([1.0])
grad_tiny = t.tensor([1e-5])   # near-zero gradient to isolate decay effect
m = t.zeros(1)
v = t.zeros(1)

trajectory = [p.item()]
for step in range(1, T + 1):
    # Decoupled weight decay
    p.mul_(1 - lr * wd)
    # Adam moment update
    m.mul_(beta1).add_(grad_tiny, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad_tiny, grad_tiny, value=1 - beta2)
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
    trajectory.append(p.item())

# Expected approximate decay from decoupled term alone
geometric = [1.0 * (1 - lr * wd) ** k for k in range(T + 1)]

print('Actual trajectory (10 steps):')
for s, (actual, expected) in enumerate(zip(trajectory, geometric)):
    print(f'  step {s}: p={actual:.6f}, geometric_approx={expected:.6f}')
print('\nParameter shrinks toward zero (decoupled decay at work):', trajectory[-1] < trajectory[0])